# 03 - Transformação da Camada Silver

## Objetivo

Este notebook transforma os dados da camada Bronze em tabelas Delta
padronizadas e preparadas para consumo analítico.

As regras implementadas foram definidas a partir do diagnóstico de
qualidade realizado na camada Bronze.

Principais objetivos:

- padronizar tipos e valores categóricos;
- tratar inconsistências identificadas;
- preservar valores extremos potencialmente legítimos;
- criar indicadores de qualidade e negócio;
- enriquecer os dados quando necessário;
- manter rastreabilidade entre Bronze e Silver;
- preparar os dados para a modelagem dimensional da camada Gold.

In [0]:
from pyspark.sql import functions as F

CATALOG = "datalake_mvp"
BRONZE = "mvp_bronze"
SILVER = "mvp_silver"

def bronze_table(tabela):
    return spark.table(f"{CATALOG}.{BRONZE}.{tabela}")

def salvar_silver(df, tabela):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SILVER}.{tabela}")
    )

    print(f"✓ {CATALOG}.{SILVER}.{tabela} criada com {df.count():,} registros")

In [0]:
df_customers = bronze_table("customers")
df_orders = bronze_table("orders")
df_items = bronze_table("order_items")
df_payments = bronze_table("order_payments")
df_reviews = bronze_table("order_reviews")
df_products = bronze_table("products")
df_sellers = bronze_table("sellers")
df_geolocation = bronze_table("geolocation")
df_translation = bronze_table("product_category_translation")

print("Tabelas Bronze carregadas com sucesso.")

Tabelas Bronze carregadas com sucesso.


## 3.1 - Transformação de Clientes

A tabela de clientes apresentou integridade adequada para a chave
`customer_id`, sem duplicidades identificadas.

Nesta etapa são realizadas padronizações textuais e de tipos, preservando
a granularidade original da entidade. Não são removidos registros, pois
não foram identificadas inconsistências que justificassem exclusão.

In [0]:
silver_customers = (
    df_customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        F.lower(F.trim(F.col("customer_city"))).alias("customer_city"),
        F.upper(F.trim(F.col("customer_state"))).alias("customer_state"),
        "_ingestion_timestamp",
        "_source_file"
    )
)

salvar_silver(silver_customers, "customers")

✓ datalake_mvp.mvp_silver.customers criada com 99,441 registros


In [0]:
bronze_count = df_customers.count()
silver_count = silver_customers.count()

print(f"Bronze customers : {bronze_count:,}")
print(f"Silver customers : {silver_count:,}")
print(f"Diferença        : {bronze_count - silver_count:,}")

Bronze customers : 99,441
Silver customers : 99,441
Diferença        : 0


## 3.2 - Transformação de Vendedores

A tabela de vendedores apresentou unicidade adequada para `seller_id`.

Foram aplicadas padronizações nos atributos textuais de cidade e estado,
mantendo todos os registros originais e os campos necessários para
posterior enriquecimento geográfico e construção da camada Gold.

In [0]:
silver_sellers = (
    df_sellers
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        F.lower(F.trim(F.col("seller_city"))).alias("seller_city"),
        F.upper(F.trim(F.col("seller_state"))).alias("seller_state"),
        "_ingestion_timestamp",
        "_source_file"
    )
)

salvar_silver(silver_sellers, "sellers")

✓ datalake_mvp.mvp_silver.sellers criada com 3,095 registros


In [0]:
bronze_count = df_sellers.count()
silver_count = silver_sellers.count()

print(f"Bronze sellers : {bronze_count:,}")
print(f"Silver sellers : {silver_count:,}")
print(f"Diferença      : {bronze_count - silver_count:,}")

Bronze sellers : 3,095
Silver sellers : 3,095
Diferença      : 0


## 3.3 - Transformação de Produtos

A análise de qualidade identificou quatro produtos com peso igual a zero,
apesar da existência de dimensões físicas válidas. Esses valores foram
considerados implausíveis e convertidos para nulo, evitando a imputação
arbitrária de um peso sem evidência na fonte.

Os demais valores ausentes foram preservados quando não havia informação
confiável para sua substituição.

A tabela também foi enriquecida com a tradução das categorias de produtos,
utilizando a tabela de referência `product_category_translation`.

Os outliers identificados pelo método IQR foram preservados, pois valores
extremos não representam necessariamente erros e podem corresponder a
produtos legítimos.

In [0]:
display(
    df_translation
    .groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
)

product_category_name,count


In [0]:
silver_products = (
    df_products.alias("p")
    .join(
        df_translation.alias("t"),
        F.col("p.product_category_name") == F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("p.product_id"),
        F.col("p.product_category_name"),
        F.col("t.product_category_name_english"),
        F.col("p.product_name_lenght").alias("product_name_length"),
        F.col("p.product_description_lenght").alias("product_description_length"),
        F.col("p.product_photos_qty"),
        
        F.when(
            F.col("p.product_weight_g") <= 0,
            F.lit(None)
        )
        .otherwise(F.col("p.product_weight_g"))
        .alias("product_weight_g"),
        
        F.col("p.product_length_cm"),
        F.col("p.product_height_cm"),
        F.col("p.product_width_cm"),
        F.col("p._ingestion_timestamp"),
        F.col("p._source_file")
    )
)

salvar_silver(silver_products, "products")

✓ datalake_mvp.mvp_silver.products criada com 32,951 registros


In [0]:
bronze_count = df_products.count()
silver_count = silver_products.count()

print(f"Bronze products : {bronze_count:,}")
print(f"Silver products : {silver_count:,}")
print(f"Diferença       : {bronze_count - silver_count:,}")

Bronze products : 32,951
Silver products : 32,951
Diferença       : 0


In [0]:
display(
    silver_products.select(
        F.sum(
            F.when(F.col("product_weight_g").isNull(), 1)
            .otherwise(0)
        ).alias("pesos_nulos_silver")
    )
)

pesos_nulos_silver
6


In [0]:
peso_zero_bronze = (
    df_products
    .filter(F.col("product_weight_g") == 0)
    .count()
)

peso_zero_silver = (
    silver_products
    .filter(F.col("product_weight_g") == 0)
    .count()
)

print(f"Pesos iguais a zero - Bronze : {peso_zero_bronze}")
print(f"Pesos iguais a zero - Silver : {peso_zero_silver}")
print(f"Registros tratados           : {peso_zero_bronze - peso_zero_silver}")

Pesos iguais a zero - Bronze : 4
Pesos iguais a zero - Silver : 0
Registros tratados           : 4


## 3.4 - Transformação de Pedidos

A tabela de pedidos foi enriquecida com indicadores de qualidade e atributos
temporais derivados.

As datas originais foram preservadas. Inconsistências cronológicas não foram
corrigidas artificialmente, pois não existem informações suficientes para
determinar qual timestamp representa o valor correto.

Foram criados indicadores para identificar:

- aprovação anterior à compra;
- envio à transportadora anterior à aprovação;
- entrega ao cliente anterior ao envio;
- existência de qualquer inconsistência temporal;
- entrega realizada após o prazo estimado.

Também foram calculados indicadores de duração do processo logístico para
posterior utilização nas análises da camada Gold.

In [0]:
silver_orders = (
    df_orders

    # 1. Padronização do status
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )

    # 2. Flags de qualidade temporal
    .withColumn(
        "flag_approval_before_purchase",
        F.when(
            F.col("order_approved_at").isNotNull() &
            (
                F.col("order_approved_at") <
                F.col("order_purchase_timestamp")
            ),
            1
        ).otherwise(0)
    )

    .withColumn(
        "flag_carrier_before_approval",
        F.when(
            F.col("order_delivered_carrier_date").isNotNull() &
            F.col("order_approved_at").isNotNull() &
            (
                F.col("order_delivered_carrier_date") <
                F.col("order_approved_at")
            ),
            1
        ).otherwise(0)
    )

    .withColumn(
        "flag_delivery_before_carrier",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_delivered_carrier_date").isNotNull() &
            (
                F.col("order_delivered_customer_date") <
                F.col("order_delivered_carrier_date")
            ),
            1
        ).otherwise(0)
    )

    # 3. Flag consolidada de qualidade temporal
    .withColumn(
        "flag_temporal_inconsistency",
        F.when(
            (F.col("flag_approval_before_purchase") == 1) |
            (F.col("flag_carrier_before_approval") == 1) |
            (F.col("flag_delivery_before_carrier") == 1),
            1
        ).otherwise(0)
    )

    # 4. Tempo entre compra e entrega
    .withColumn(
        "delivery_time_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.to_date(F.col("order_delivered_customer_date")),
                F.to_date(F.col("order_purchase_timestamp"))
            )
        )
    )

    # 5. Diferença entre entrega real e estimada
    .withColumn(
        "delivery_delay_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_estimated_delivery_date").isNotNull(),
            F.datediff(
                F.to_date(F.col("order_delivered_customer_date")),
                F.to_date(F.col("order_estimated_delivery_date"))
            )
        )
    )

    # 6. Flag de atraso baseada na granularidade diária
    .withColumn(
        "flag_late_delivery",
        F.when(
            F.col("delivery_delay_days").isNull(),
            F.lit(0)
        )
        .when(
            F.col("delivery_delay_days") > 0,
            F.lit(1)
        )
        .otherwise(F.lit(0))
    )
)

In [0]:
salvar_silver(silver_orders, "orders")

✓ datalake_mvp.mvp_silver.orders criada com 99,441 registros


In [0]:
display(
    silver_orders.select(
        F.sum("flag_approval_before_purchase")
            .alias("approval_before_purchase"),

        F.sum("flag_carrier_before_approval")
            .alias("carrier_before_approval"),

        F.sum("flag_delivery_before_carrier")
            .alias("delivery_before_carrier"),

        F.sum("flag_temporal_inconsistency")
            .alias("orders_with_temporal_inconsistency"),

        F.sum("flag_late_delivery")
            .alias("late_deliveries")
    )
)

approval_before_purchase,carrier_before_approval,delivery_before_carrier,orders_with_temporal_inconsistency,late_deliveries
0,1359,23,1382,6535


In [0]:
display(
    silver_orders.select(
        F.min("delivery_time_days").alias("min_delivery_days"),
        F.avg("delivery_time_days").alias("avg_delivery_days"),
        F.max("delivery_time_days").alias("max_delivery_days"),

        F.min("delivery_delay_days").alias("min_delay_days"),
        F.avg("delivery_delay_days").alias("avg_delay_days"),
        F.max("delivery_delay_days").alias("max_delay_days")
    )
)

min_delivery_days,avg_delivery_days,max_delivery_days,min_delay_days,avg_delay_days,max_delay_days
0,12.497336125046644,210,-147,-11.876881296902857,188


## 3.5 - Transformação de Itens do Pedido

A tabela de itens apresentou integridade adequada para a chave composta
`order_id + order_item_id`, sem duplicidades identificadas.

Também não foram encontrados valores negativos nos atributos de preço
e frete.

Os valores extremos identificados pelo método IQR foram preservados, pois
podem representar transações legítimas. Nesta etapa são mantidos os atributos
necessários para posterior construção das métricas financeiras e comerciais
da camada Gold.

In [0]:
silver_order_items = (
    df_items
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        F.col("price").cast("double").alias("price"),
        F.col("freight_value").cast("double").alias("freight_value"),

        # Valor total do item considerando produto + frete
        (
            F.col("price") + F.col("freight_value")
        ).cast("double").alias("item_total_value"),

        "_ingestion_timestamp",
        "_source_file"
    )
)

salvar_silver(silver_order_items, "order_items")

✓ datalake_mvp.mvp_silver.order_items criada com 112,650 registros


In [0]:
bronze_count = df_items.count()
silver_count = silver_order_items.count()

print(f"Bronze order_items : {bronze_count:,}")
print(f"Silver order_items : {silver_count:,}")
print(f"Diferença          : {bronze_count - silver_count:,}")

Bronze order_items : 112,650
Silver order_items : 112,650
Diferença          : 0


In [0]:
display(
    silver_order_items.select(
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.min("freight_value").alias("min_freight"),
        F.max("freight_value").alias("max_freight"),
        F.min("item_total_value").alias("min_item_total"),
        F.max("item_total_value").alias("max_item_total")
    )
)

min_price,max_price,min_freight,max_freight,min_item_total,max_item_total
0.85,6735.0,0.0,409.68,6.08,6929.31


## 3.6 - Transformação de Pagamentos

A tabela de pagamentos possui múltiplos registros por pedido, refletindo
a possibilidade de diferentes sequências ou formas de pagamento.

Foram identificados três registros com tipo de pagamento `not_defined`,
todos com valor igual a zero. Esses registros foram preservados e a
categoria foi padronizada para `unknown`, evitando perda de informação.

Não foram identificados valores negativos de pagamento ou quantidade
negativa de parcelas. Os valores extremos identificados pelo método IQR
também foram preservados.

In [0]:
silver_payments = (
    df_payments
    .select(
        "order_id",
        "payment_sequential",

        F.when(
            F.lower(F.trim(F.col("payment_type"))) == "not_defined",
            "unknown"
        )
        .otherwise(
            F.lower(F.trim(F.col("payment_type")))
        )
        .alias("payment_type"),

        "payment_installments",

        F.col("payment_value")
        .cast("double")
        .alias("payment_value"),

        "_ingestion_timestamp",
        "_source_file"
    )
)

salvar_silver(silver_payments, "order_payments")

✓ datalake_mvp.mvp_silver.order_payments criada com 103,886 registros


In [0]:
bronze_count = df_payments.count()
silver_count = silver_payments.count()

print(f"Bronze order_payments : {bronze_count:,}")
print(f"Silver order_payments : {silver_count:,}")
print(f"Diferença             : {bronze_count - silver_count:,}")

Bronze order_payments : 103,886
Silver order_payments : 103,886
Diferença             : 0


In [0]:
display(
    silver_payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("quantidade"),
        F.round(F.sum("payment_value"), 2).alias("valor_total")
    )
    .orderBy(F.desc("quantidade"))
)

payment_type,quantidade,valor_total
credit_card,76795,1.254208419E7
boleto,19784,2869361.27
voucher,5775,379436.87
debit_card,1529,217989.79
unknown,3,0.0


## 3.7 - Transformação de Avaliações

A tabela de avaliações contém atributos textuais opcionais, apresentando
quantidade significativa de valores nulos nos campos de título e comentário.
Esses valores foram preservados, pois a ausência de texto não caracteriza
necessariamente um problema de qualidade.

Durante a ingestão foi identificado e corrigido um problema de parsing
relacionado a campos textuais multilinha. Após a correção, os atributos
estruturais da tabela passaram a apresentar consistência adequada.

As notas de avaliação respeitam o domínio esperado entre 1 e 5. Os registros
foram preservados em sua granularidade original, sem remoção automática de
avaliações repetidas.

In [0]:
silver_reviews = (
    df_reviews
    .select(
        "review_id",
        "order_id",

        F.col("review_score")
        .cast("int")
        .alias("review_score"),

        F.when(
            F.trim(F.col("review_comment_title")) == "",
            F.lit(None)
        )
        .otherwise(F.trim(F.col("review_comment_title")))
        .alias("review_comment_title"),

        F.when(
            F.trim(F.col("review_comment_message")) == "",
            F.lit(None)
        )
        .otherwise(F.trim(F.col("review_comment_message")))
        .alias("review_comment_message"),

        "review_creation_date",
        "review_answer_timestamp",

        "_ingestion_timestamp",
        "_source_file"
    )
)

salvar_silver(silver_reviews, "order_reviews")

✓ datalake_mvp.mvp_silver.order_reviews criada com 99,224 registros


In [0]:
bronze_count = df_reviews.count()
silver_count = silver_reviews.count()

print(f"Bronze order_reviews : {bronze_count:,}")
print(f"Silver order_reviews : {silver_count:,}")
print(f"Diferença            : {bronze_count - silver_count:,}")

Bronze order_reviews : 99,224
Silver order_reviews : 99,224
Diferença            : 0


## 3.8 - Transformação de Geolocalização

A tabela de geolocalização possui múltiplos registros para um mesmo prefixo
de CEP. Dessa forma, o atributo `geolocation_zip_code_prefix` não representa
uma chave única na camada Bronze.

Para permitir seu uso seguro no enriquecimento de clientes e vendedores,
a tabela foi consolidada para uma linha por prefixo de CEP.

Latitude e longitude foram agregadas pela média das coordenadas disponíveis.
Para cidade e estado foram utilizadas representações padronizadas associadas
ao prefixo de CEP.

Essa transformação reduz a granularidade da tabela propositalmente e evita
multiplicação indevida de registros em operações de JOIN posteriores.

In [0]:
geo_base = (
    df_geolocation
    .select(
        "geolocation_zip_code_prefix",

        F.col("geolocation_lat")
        .cast("double")
        .alias("geolocation_lat"),

        F.col("geolocation_lng")
        .cast("double")
        .alias("geolocation_lng"),

        F.lower(
            F.trim(F.col("geolocation_city"))
        ).alias("geolocation_city"),

        F.upper(
            F.trim(F.col("geolocation_state"))
        ).alias("geolocation_state")
    )
)

In [0]:
silver_geolocation = (
    geo_base
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat")
        .alias("geolocation_lat"),

        F.avg("geolocation_lng")
        .alias("geolocation_lng"),

        F.first(
            "geolocation_city",
            ignorenulls=True
        ).alias("geolocation_city"),

        F.first(
            "geolocation_state",
            ignorenulls=True
        ).alias("geolocation_state"),

        F.count("*")
        .alias("geolocation_source_records")
    )
)

In [0]:
salvar_silver(
    silver_geolocation,
    "geolocation"
)

✓ datalake_mvp.mvp_silver.geolocation criada com 19,015 registros


In [0]:
bronze_count = df_geolocation.count()
silver_count = silver_geolocation.count()

print(f"Bronze geolocation : {bronze_count:,}")
print(f"Silver geolocation : {silver_count:,}")
print(f"Redução de registros: {bronze_count - silver_count:,}")

Bronze geolocation : 1,000,163
Silver geolocation : 19,015
Redução de registros: 981,148


In [0]:
display(
    silver_geolocation
    .groupBy("geolocation_zip_code_prefix")
    .count()
    .filter(F.col("count") > 1)
)

geolocation_zip_code_prefix,count


## 3.9 - Validação da Camada Silver

Após as transformações, foi realizada uma etapa final de validação da camada
Silver com o objetivo de verificar integridade, granularidade e consistência
dos dados preparados.

Foram avaliados:

- disponibilidade das tabelas transformadas;
- quantidade de registros;
- unicidade das principais chaves de negócio;
- integridade de atributos obrigatórios;
- domínio das avaliações;
- consistência de valores monetários;
- unicidade da geolocalização consolidada.

As validações críticas foram implementadas de forma programática, permitindo
que inconsistências interrompam a execução do processo antes da construção
da camada Gold.

In [0]:
# ============================================================
# 3.9 - Validação da Camada Silver
# ============================================================

validacoes = []

def registrar_teste(teste, resultado, detalhe):
    validacoes.append(
        (
            teste,
            "OK" if resultado else "ERRO",
            detalhe
        )
    )

In [0]:
# 1. Customers - chave única

total = silver_customers.count()
distintos = silver_customers.select("customer_id").distinct().count()

registrar_teste(
    "Unicidade customer_id",
    total == distintos,
    f"{total:,} registros / {distintos:,} chaves distintas"
)


# 2. Sellers - chave única

total = silver_sellers.count()
distintos = silver_sellers.select("seller_id").distinct().count()

registrar_teste(
    "Unicidade seller_id",
    total == distintos,
    f"{total:,} registros / {distintos:,} chaves distintas"
)


# 3. Products - chave única

total = silver_products.count()
distintos = silver_products.select("product_id").distinct().count()

registrar_teste(
    "Unicidade product_id",
    total == distintos,
    f"{total:,} registros / {distintos:,} chaves distintas"
)


# 4. Orders - chave única

total = silver_orders.count()
distintos = silver_orders.select("order_id").distinct().count()

registrar_teste(
    "Unicidade order_id",
    total == distintos,
    f"{total:,} registros / {distintos:,} chaves distintas"
)

In [0]:
# 5. Reviews - domínio das notas

reviews_invalidas = (
    silver_reviews
    .filter(
        (~F.col("review_score").between(1, 5)) |
        F.col("review_score").isNull()
    )
    .count()
)

registrar_teste(
    "Domínio review_score",
    reviews_invalidas == 0,
    f"{reviews_invalidas:,} avaliações fora do domínio 1-5"
)


# 6. Order Items - preços negativos

items_invalidos = (
    silver_order_items
    .filter(
        (F.col("price") < 0) |
        (F.col("freight_value") < 0)
    )
    .count()
)

registrar_teste(
    "Valores monetários order_items",
    items_invalidos == 0,
    f"{items_invalidos:,} registros com preço/frete negativo"
)


# 7. Payments - valores negativos

payments_invalidos = (
    silver_payments
    .filter(F.col("payment_value") < 0)
    .count()
)

registrar_teste(
    "Valores monetários payments",
    payments_invalidos == 0,
    f"{payments_invalidos:,} pagamentos negativos"
)


# 8. Geolocation - CEP único

total_geo = silver_geolocation.count()

distintos_geo = (
    silver_geolocation
    .select("geolocation_zip_code_prefix")
    .distinct()
    .count()
)

registrar_teste(
    "Unicidade geolocation_zip_code_prefix",
    total_geo == distintos_geo,
    f"{total_geo:,} registros / {distintos_geo:,} CEPs distintos"
)

In [0]:
df_validacoes = spark.createDataFrame(
    validacoes,
    ["teste", "status", "detalhe"]
)

display(df_validacoes)

teste,status,detalhe
Unicidade customer_id,OK,"99,441 registros / 99,441 chaves distintas"
Unicidade seller_id,OK,"3,095 registros / 3,095 chaves distintas"
Unicidade product_id,OK,"32,951 registros / 32,951 chaves distintas"
Unicidade order_id,OK,"99,441 registros / 99,441 chaves distintas"
Domínio review_score,OK,0 avaliações fora do domínio 1-5
Valores monetários order_items,OK,0 registros com preço/frete negativo
Valores monetários payments,OK,0 pagamentos negativos
Unicidade geolocation_zip_code_prefix,OK,"19,015 registros / 19,015 CEPs distintos"


In [0]:
erros = [
    teste
    for teste, status, detalhe in validacoes
    if status == "ERRO"
]

if erros:
    raise Exception(
        "Falha na validação da camada Silver: "
        + ", ".join(erros)
    )

print("✓ Camada Silver validada com sucesso.")
print(f"✓ {len(validacoes)} testes executados.")
print("✓ Nenhuma inconsistência crítica identificada.")

✓ Camada Silver validada com sucesso.
✓ 8 testes executados.
✓ Nenhuma inconsistência crítica identificada.
